In [1]:
import sys, platform, psutil, os
print("Python", sys.version.split()[0], platform.platform())
vm = psutil.virtual_memory()
print(f"RAM total {vm.total/1e9:.1f} GB, available {vm.available/1e9:.1f} GB")
print("CPU cores", os.cpu_count())
print(f"Disk free {psutil.disk_usage('/').free/1e9:.0f} GB")

Python 3.13.18 Linux-6.6.159+-x86_64-with-glibc2.39
RAM total 13.6 GB, available 12.7 GB
CPU cores 2
Disk free 94 GB


In [2]:
!pip -q install lifelines
import numpy, pandas, scipy, sklearn, statsmodels, matplotlib, lifelines
for m in (numpy, pandas, scipy, sklearn, statsmodels, matplotlib, lifelines):
    print(m.__name__, m.__version__)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
numpy 2.1.3
pandas 2.2.3
scipy 1.16.3
sklearn 1.6.1
statsmodels 0.15.0
matplotlib 3.10.0
lifelines 0.30.3


In [3]:
import numpy as np, time
from sklearn.metrics import average_precision_score
rng = np.random.default_rng(20261021)
n_genes, K = 15000, 60
y = np.zeros(n_genes, int); y[rng.choice(n_genes, K, replace=False)] = 1
def make(d):
    s = rng.normal(size=n_genes); s[y == 1] += d; return s
sA, sB = make(0.0), make(0.8)
prev = y.mean()
print("AP-lift noise method:", round(average_precision_score(y, sA)/prev, 2),
      "| planted-signal method:", round(average_precision_score(y, sB)/prev, 2))
pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]
t = time.time()
for _ in range(200):
    idx = np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))])
    average_precision_score(y[idx], sA[idx]); average_precision_score(y[idx], sB[idx])
dt = time.time() - t
print(f"200 resamples took {dt:.1f}s, so 10,000 would take about {dt*50/60:.1f} min")

AP-lift noise method: 1.22 | planted-signal method: 3.65
200 resamples took 2.5s, so 10,000 would take about 2.1 min


In [4]:
import scipy.sparse as sp
X = sp.random(50000, 20000, density=0.03, format='csr', random_state=1, dtype=np.float32)
print("Matrix size GB:", round((X.data.nbytes + X.indices.nbytes + X.indptr.nbytes)/1e9, 2))
print("RAM available now GB:", round(psutil.virtual_memory().available/1e9, 1))
t = time.time(); m = np.asarray(X.mean(axis=0)).ravel()
print("Gene means over 50,000 cells took", round(time.time() - t, 2), "s")

Matrix size GB: 0.24
RAM available now GB: 12.0
Gene means over 50,000 cells took 0.26 s


In [6]:
import subprocess
open("colab_environment_check.txt", "w").write(subprocess.run(["pip", "freeze"], capture_output=True, text=True).stdout)
from google.colab import files
files.download("colab_environment_check.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>